In [ ]:
!nvidia-smi

In [ ]:
!pip install /kaggle/input/bird-clef-2023-addones/onnxruntime-1.14.0-cp37-cp37m-manylinux_2_27_x86_64.whl --no-deps

In [ ]:
!pip list | grep onnx

In [ ]:
import sys
sys.path.append('/kaggle/input/bird-clef-2023-code/main_folder/main_folder/')

In [ ]:
import pandas as pd
import numpy as np
import librosa
import seaborn as sns
import os
import json
import IPython.display as ipd
import soundfile as sf
import torch
import h5py
import onnxruntime as ort

from glob import glob
from tqdm import tqdm
from matplotlib import pyplot as plt
from itertools import chain
from os.path import join as pjoin
from copy import deepcopy


from code_base.models import WaveCNNClasifier, WaveCNNAttenClasifier
from code_base.datasets import WaveDataset, WaveAllFileDataset
from code_base.utils.inference_utils import apply_avarage_weights_on_swa_path
from code_base.inefernce import BirdsInference
from code_base.utils import load_json, compose_submission_dataframe, groupby_np_array, stack_and_max_by_samples
from code_base.utils.metrics import padded_cmap_numpy
%matplotlib inline


# Config

In [ ]:
EXP_NAME = "convnext_small_fb_in22k_ft_in1k_384__convnextv2_tiny_fcmae_ft_in22k_in1k_384__eca_nfnet_l0_noval_v32"
TRAIN_PERIOD = 5
print("Possible checkpoints:\n\n{}".format("\n".join(set([
    os.path.basename(el) for el in glob(f"/kaggle/input/bird-clef-2023-models/{EXP_NAME}/{EXP_NAME}/*/checkpoints/*.pt*") if "train" not in os.path.basename(el)
]))))

In [ ]:
CONFIG = {
    # Main
    "run_validation": False,
    "run_test": True,
    # Inference Class
    "use_sigmoid": False,
    # Data config
#     "train_df_path":"/home/vova/data/exps/BirdCLEF_2023/birdclef_2023/train_metadata_extended.csv",
#     "split_path":"/home/vova/data/exps/BirdCLEF_2023/cv_split_2023_v1.npy",
    "folds":[0],
#     "train_data_root":"/home/vova/data/exps/BirdCLEF_2023/birdclef_2023/train_audio/",
#     "test_data_root": "/kaggle/input/bird-clef-2023-addones/fake_test_20/fake_test_20/*.ogg",
    "test_data_root": "/kaggle/input/birdclef-2023/test_soundscapes/*.ogg",
    "label_map_data_path":'/kaggle/input/bird-clef-2023-models/bird2int_2023.json',
#     "label_map_data_path": "/kaggle/input/bird-clef-2023-models/bird2id_v3.json",
#     "label_map_data_path":'/kaggle/input/bird-clef-2023-models/bird221int_202x.json',
#     "label_map_data_path": "/kaggle/input/bird-clef-2023-models/xc_birds_202x_only_scored.json",
#     "label_map_data_path": '/kaggle/input/bird-clef-2023-models/bird2id_xc_pretrain.json',
#     "label_map_data_path": "/kaggle/input/bird-clef-2023-models/bird2id.json",
#     "label_map_data_path":"/kaggle/input/bird-clef-2023-models/bird2id_v1.json",
    "lookback":None,
    "lookahead":None,
    "segment_len":5,
    "step": None,
    "late_normalize": True,
    # Model config
    "exp_name":EXP_NAME,
#     "model_class": WaveCNNAttenClasifier,
#     "model_config": dict(
#         backbone="convnext_tiny_in22ft1k",
#         mel_spec_paramms={
#             "sample_rate": 32000,
#             "n_mels": 128,
#             "f_min": 20,
#             "n_fft": 2048,
#             "hop_length": 512,
#             "normalized": True,
#         },
#         head_config={
#             "p": 0.5,
#             "num_class": 264,
#             "train_period": TRAIN_PERIOD,
#             "infer_period": TRAIN_PERIOD,
#         },
#         pretrained=False
#     ),
#     "chkp_name":"model.last.pth",
#     "swa_checkpoint": None,
#     "distributed_chkp": False,
}

if CONFIG.get("use_sed_mode", False):
    assert CONFIG["step"] is not None
else:
    assert CONFIG["step"] is None
    
if "folds" not in CONFIG:
    CONFIG["folds"] = list(range(CONFIG["n_folds"]))

# Data

In [ ]:
bird2id = load_json(CONFIG["label_map_data_path"])

In [ ]:
if CONFIG["run_validation"]:
    df = pd.read_csv(CONFIG["train_df_path"])
    split = np.load(CONFIG["split_path"], allow_pickle=True)
    val_df = [df.iloc[split[fold_id][1]].reset_index(drop=True) for fold_id in CONFIG["folds"]]

In [ ]:
if CONFIG["run_test"]:
    test_au_pathes = glob(CONFIG["test_data_root"])

    test_df = pd.DataFrame({
        "filename": test_au_pathes,
        "duration_s": [librosa.get_duration(filename=el) for el in test_au_pathes]
    })

In [ ]:
if CONFIG["run_validation"]:
    val_ds_conig = {
       "root": CONFIG["train_data_root"],
       "label_str2int_mapping_path": CONFIG["label_map_data_path"],
       "use_audio_cache": True,
       "n_cores": 64,
       "verbose": False,
       "segment_len": CONFIG["segment_len"],
       "lookback":CONFIG["lookback"],
       "lookahead":CONFIG["lookahead"],
       "sample_id": None,
       "late_normalize": CONFIG["late_normalize"],
       "step": CONFIG["step"],
       "validate_sr": 32_000
    }
if CONFIG["run_test"]:
    ds_config_test = {
       "root": "",
       "label_str2int_mapping_path": CONFIG["label_map_data_path"],
       "n_cores": 64,
       "use_audio_cache": True,
       "test_mode": True,
       "segment_len": CONFIG["segment_len"],
       "lookback":CONFIG["lookback"],
       "lookahead":CONFIG["lookahead"],
        "sample_id": None,
        "late_normalize": CONFIG["late_normalize"],
        "step": CONFIG["step"],
        "validate_sr": 32_000
    }
loader_config = {
    "batch_size": 4,
    "drop_last": False,
    "shuffle": False,
    "num_workers": 0,
}

In [ ]:
if CONFIG["run_test"]:
    ds_test = WaveAllFileDataset(df=test_df, **ds_config_test)
if CONFIG["run_validation"]:
    ds_val = [WaveAllFileDataset(df=df, **val_ds_conig) for df in val_df]

In [ ]:
if CONFIG["run_validation"]:
    loader_val = [torch.utils.data.DataLoader(
        ds,
        **loader_config,
    )for ds in ds_val]
if CONFIG["run_test"]:
    loader_test = torch.utils.data.DataLoader(
        ds_test,
        **loader_config,
    )

# Model

In [ ]:
def create_model_and_upload_chkp(
    model_class,
    model_config,
    model_device,
    model_chkp,
    use_distributed=False,
    swa_checkpoint=None
):
    print(model_chkp)
    if "swa" in model_chkp:
        print("swa by {}".format(os.path.splitext(os.path.basename(model_chkp))[0]))
        t_chkp = apply_avarage_weights_on_swa_path(model_chkp, use_distributed=use_distributed, take_best=swa_checkpoint)
    else:
        print("vanilla model")
        t_chkp = torch.load(model_chkp, map_location="cpu")
        
    t_model = model_class(**model_config, device=model_device)
    t_model.load_state_dict(t_chkp)
    t_model.eval()
    return t_model

In [ ]:
# model = [create_model_and_upload_chkp(
#         model_class=CONFIG["model_class"],
#         model_config=CONFIG['model_config'],
#         model_device="cpu",
#         model_chkp=f"/kaggle/input/bird-clef-2023-models/{CONFIG['exp_name']}/{CONFIG['exp_name']}/fold_{m_i}/checkpoints/{CONFIG['chkp_name']}",
#         swa_checkpoint=CONFIG['swa_checkpoint'],
#         use_distributed=CONFIG['distributed_chkp']
# ) for m_i in CONFIG["folds"]]

In [ ]:
model = ort.InferenceSession(f"/kaggle/input/bird-clef-2023-models/{CONFIG['exp_name']}/{CONFIG['exp_name']}/checkpoints/model_simpl.onnx")

# Inference Class

In [ ]:
inference_class = BirdsInference(
    device="cpu",
    verbose_tqdm=True,
    use_sigmoid=CONFIG["use_sigmoid"],
#     model_output_key=CONFIG["model_output_key"],
)

# Val Predict

In [ ]:
if CONFIG["run_validation"]:
    val_tgts, val_preds, val_preds_long = inference_class.predict_val_loaders(
        nn_models=model,
        data_loaders=loader_val
    )
    print(
        f"Min prob {val_preds.min()}. Max prob {val_preds.max()}.\n"
        f"Padded CMAP: {padded_cmap_numpy(val_tgts, val_preds)}"
    )

# Test Pred

In [ ]:
if CONFIG["run_test"]:
    test_preds, test_preds_long, test_dfidx, test_end = inference_class.predict_test_loader(
        nn_models=model,
        data_loader=loader_test,
        is_onnx_model=True
    )
    test_pred_df = compose_submission_dataframe(
        probs=test_preds,
        dfidxs=test_dfidx,
        end_seconds=test_end,
        filenames=loader_test.dataset.df[loader_test.dataset.name_col].copy(),
        bird2id=bird2id,
    )
    sample_submission = pd.read_csv("/kaggle/input/birdclef-2023/sample_submission.csv")
    if test_pred_df.shape[1] > sample_submission.shape[1]:
        print("Shrinking columns")
        test_pred_df = test_pred_df[sample_submission.columns]

In [ ]:
test_pred_df.iloc[:,1:].values.max(), test_pred_df.iloc[:,1:].values.min()

In [ ]:
plt.hist(test_pred_df.iloc[:,1:].values.max(axis=1), bins=30);

# Map Predictions

In [ ]:
if CONFIG.get("check_inf_class", False):
    val_ds_check = WaveAllFileDataset(df=val_df[0], **{
           "root": CONFIG["train_data_root"],
           "label_str2int_mapping_path": CONFIG["label_map_data_path"],
           "n_cores": 64,
           "use_audio_cache": True,
           "test_mode": True,
           "segment_len": CONFIG["segment_len"],
           "lookback":CONFIG["lookback"],
           "lookahead":CONFIG["lookahead"],
            "sample_id": None,
            "late_normalize": CONFIG["late_normalize"],
            "step": CONFIG["step"],
        }
    )
    val_loader_check = torch.utils.data.DataLoader(
        val_ds_check,
        **loader_config
    )
    
    test_preds_check, test_preds_long_check, test_dfidx_check, test_end_check = inference_class.predict_test_loader(
        nn_models=model,
        data_loader=val_loader_check
    )
    test_preds_check_grouped = groupby_np_array(
        groupby_f=test_dfidx_check,
        array_to_group=test_preds_check,
        apply_f=stack_and_max_by_samples,
    )
    print(np.allclose(
        test_preds_check_grouped,
        val_preds
    ))

# Boost Probs

In [ ]:
# CLASSES = list(set(test_pred_df.columns[1:]))
# print(len(CLASSES))

In [ ]:
# def connected_region_indices(bool_array):
#     connected_regions = []
#     region = []

#     for i, value in enumerate(bool_array):
#         if value:
#             region.append(i)
#         elif region:
#             connected_regions.append(region)
#             region = []

#     if region:
#         connected_regions.append(region)

#     return connected_regions

# def modify_probabilities(group, lower_prob=0.5, max_prob=0.75, min_seq_len=2):
#     class_cols = ['shesta1', 'colsun2', 'amesun2', 'bcbeat1', 'marsun2']
    
#     mask_low = (group[CLASSES] > lower_prob).values
#     mask_high = (group[CLASSES] > max_prob).values
    
#     # At least 2 chunks AND exceeds max_prob
#     classes_to_boost = np.where((mask_low.sum(axis=0) >= min_seq_len) & mask_high.any(axis=0))[0]
#     if len(classes_to_boost) > 0:
#         for cls in classes_to_boost:
#             new_probs = group[CLASSES[cls]].values.copy()
#             connected_regions = connected_region_indices(mask_low[:,cls])
#             for region in connected_regions:
#                 if len(region) >= min_seq_len:
#                     max_region_prob = group[CLASSES[cls]].iloc[region].max()
#                     if max_region_prob > max_prob:
#                         # print(f"Boosting: {CLASSES[cls]}")
#                         new_probs[region] = max_region_prob
#             group[CLASSES[cls]] = new_probs
                
#     return group

In [ ]:
# test_pred_df["id"] = test_pred_df["row_id"].apply(lambda x: "_".join(x.split("_")[:-1]))
# test_pred_df["sec"] = test_pred_df["row_id"].apply(lambda x: int(x.split("_")[-1]))

# test_pred_df = test_pred_df.sort_values(["id", "sec"]).reset_index(drop=True)

# test_pred_df = test_pred_df.groupby("id").apply(modify_probabilities).reset_index(drop=True)

# test_pred_df = test_pred_df.drop(columns=["id", "sec"])

In [ ]:
# np.where((test_pred_df_new[CLASSES].values != test_pred_df[CLASSES].values).sum(axis=0))
# test_pred_df_new.loc[(test_pred_df_new[CLASSES[185]] - test_pred_df[CLASSES[185]]) > 0, CLASSES[185]]
# test_pred_df.loc[(test_pred_df_new[CLASSES[185]] - test_pred_df[CLASSES[185]]) > 0, CLASSES[185]]

In [ ]:
# def boost_pred_prob(
#     input_df,
#     class_columns,
#     select_prob=0.8,
#     boost_prob=0.2
# ):
#     result_df = input_df.copy()
#     for sample_id in tqdm(set(input_df["id"])):
#         sample_id_pred_df = result_df.loc[result_df["id"] == sample_id, class_columns]
#         boost_classes = class_columns[sample_id_pred_df.values.max(axis=0) > select_prob]
#         boost_mask = sample_id_pred_df[boost_classes].values < select_prob - boost_prob
#         result_df.loc[result_df["id"] == sample_id, boost_classes] = (
#             (boost_mask.astype(np.float32) * (sample_id_pred_df[boost_classes].values + boost_prob)) +
#             ((~boost_mask).astype(np.float32) * sample_id_pred_df[boost_classes].values)
#         )
#     return result_df

# test_pred_df["id"] = test_pred_df["row_id"].apply(lambda x: "_".join(x.split("_")[:-1]))
# test_pred_df["sec"] = test_pred_df["row_id"].apply(lambda x: int(x.split("_")[-1]))

# test_pred_df = boost_pred_prob(
#     input_df=test_pred_df,
#     class_columns=test_pred_df.columns[1:-2]
# )
# test_pred_df = test_pred_df.drop(columns=["id", "sec"])

In [ ]:
# test_pred_df.iloc[:,1:].values.max(), test_pred_df.iloc[:,1:].values.min()

In [ ]:
# plt.hist(test_pred_df.iloc[:,1:].values.max(axis=1), bins=30);

# Save Prediction

In [ ]:
sample_submission = pd.read_csv("/kaggle/input/birdclef-2023/sample_submission.csv")
assert set(sample_submission.columns) == set(test_pred_df.columns)
test_pred_df = test_pred_df[sample_submission.columns]
test_pred_df.to_csv("submission.csv", index=False)